### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [1]:
!pip install --upgrade langchain


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install -U langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [4]:
import langchain
print(langchain.__version__)


1.2.14


In [5]:
from pathlib import Path

def process_all_pdfs(pdf_directory):
    """
    Load all PDFs in a directory (recursively) and return chunked documents.
    Works for 1 PDF or multiple PDFs.
    """
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF file(s) to process.")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source info to metadata for citations later
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

    
pdf_documents = process_all_pdfs("../data")
    

Found 1 PDF file(s) to process.

Processing: Datafile.pdf

Total documents loaded: 90


In [6]:
pdf_documents

[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-03-31T15:48:36+05:30', 'source': '..\\data\\Datafile.pdf', 'file_path': '..\\data\\Datafile.pdf', 'total_pages': 90, 'format': 'PDF 1.7', 'title': '', 'author': 'Mounisha Anisetti', 'subject': '', 'keywords': '', 'moddate': '2026-03-31T15:48:36+05:30', 'trapped': '', 'modDate': "D:20260331154836+05'30'", 'creationDate': "D:20260331154836+05'30'", 'page': 0, 'source_file': 'Datafile.pdf', 'file_type': 'pdf'}, page_content='# HR POLICY MANUAL - COMPLETE DOCUMENT FOR RAG MODEL TRAINING \n## Version 1.0 - Anonymized Generic Policies \n \n--- \n \n## TABLE OF CONTENTS \n1. Attendance Policy \n2. Leave Policy   \n3. Salary & Compensation Policy \n4. Travel & Expenses Policy \n5. Employment Terms & Conditions \n6. Performance Management Policy \n7. Disciplinary Conduct Policy \n8. Employee Exit Policy \n9. Grievance Resolution Policy \n10. Training & Development Policy \n \n--- \n 

### Chunking

In [7]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """
    Split documents into chunks using RecursiveCharacterTextSplitter.
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""])
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    #Example
    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")  # Print first 200 characters of the first chunk
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs
chunks = split_documents(pdf_documents)
chunks

Split 90 documents into 119 chunks.

Example chunk:
Content: # HR POLICY MANUAL - COMPLETE DOCUMENT FOR RAG MODEL TRAINING 
## Version 1.0 - Anonymized Generic Policies 
 
--- 
 
## TABLE OF CONTENTS 
1. Attendance Policy 
2. Leave Policy   
3. Salary & Compens...
Metadata: {'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-03-31T15:48:36+05:30', 'source': '..\\data\\Datafile.pdf', 'file_path': '..\\data\\Datafile.pdf', 'total_pages': 90, 'format': 'PDF 1.7', 'title': '', 'author': 'Mounisha Anisetti', 'subject': '', 'keywords': '', 'moddate': '2026-03-31T15:48:36+05:30', 'trapped': '', 'modDate': "D:20260331154836+05'30'", 'creationDate': "D:20260331154836+05'30'", 'page': 0, 'source_file': 'Datafile.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-03-31T15:48:36+05:30', 'source': '..\\data\\Datafile.pdf', 'file_path': '..\\data\\Datafile.pdf', 'total_pages': 90, 'format': 'PDF 1.7', 'title': '', 'author': 'Mounisha Anisetti', 'subject': '', 'keywords': '', 'moddate': '2026-03-31T15:48:36+05:30', 'trapped': '', 'modDate': "D:20260331154836+05'30'", 'creationDate': "D:20260331154836+05'30'", 'page': 0, 'source_file': 'Datafile.pdf', 'file_type': 'pdf'}, page_content='# HR POLICY MANUAL - COMPLETE DOCUMENT FOR RAG MODEL TRAINING \n## Version 1.0 - Anonymized Generic Policies \n \n--- \n \n## TABLE OF CONTENTS \n1. Attendance Policy \n2. Leave Policy   \n3. Salary & Compensation Policy \n4. Travel & Expenses Policy \n5. Employment Terms & Conditions \n6. Performance Management Policy \n7. Disciplinary Conduct Policy \n8. Employee Exit Policy \n9. Grievance Resolution Policy \n10. Training & Development Policy \n \n--- \n 

### Embedding generation and Vector store DB

In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

sentence_transformer because the embedding model is here
uuid is for every record we are inserting in the vectordb has spme kind of id to generate that

In [ ]:
class EmbeddingManager:
    #Handles document embedding generation using SentenceTransformer and manages vector store operations with ChromaDB
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        #Initialize Embedding manager
        #Args: model_name:HuggingFace model name for sentence embedding generation
        self.model_name = model_name
        self.model=None
        self._load_model()

        #protected fnx has __ prefixe.g. __load_model() to indicate it's for internal use only and not part of the public API of the class. It's a convention to signal that this method is intended to be used only within the class and should not be accessed directly from outside the class.
    def _load_model(self):
        #lod the sentence transformer model
        try:
            print(f"Loaded embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
            
        except Exception as e:
            print(f"Error loading model {self.model}: {e}")
            raise 
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        #Generate embeddings for a list of texts
        #Args: texts:List of strings to generate embeddings for
        #Returns: Numpy array of embeddings with shape (len(texts), embedding_dim)
        if not self.model:
            raise ValueError("Embedding model is not loaded.")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    """def get_embedding_dimension(self) -> int:
        #Get the dimension of the embeddings generated by the model
        if not self.model:
            raise ValueError("Embedding model is not loaded.")
        return self.model.get_sentence_embedding_dimension()""" 


##Initialize embedding manager and generate embeddings for document chunks
embedding_manager = EmbeddingManager()
embedding_manager

Loaded embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


###     Vector store

In [10]:
from typing import List
from langchain_core.documents import Document
import os

persist_directory is where we vector storage in hard disk 

In [11]:
class VectorStore:
    #Manages vector store operations using ChromaDB for storing and retrieving document embeddings
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        #Initialize vector store
        #Args: collection_name: Name of the ChromaDB collection to use for storing document chunks and embeddings
        #persist_directory: Directory where the ChromaDB collection will be persisted on disk
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        #Initialize ChromaDB client and collection
        try:
            #create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            #get or create collection for storing document chunks and embeddings
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Collection of PDF document chunks and embeddings for RAG system"}
            )
            print(f"Vector store initialized with collection: {self.collection_name}")
            print(f"Existing documents in collection:{self.collection.count()}")  # Print number of existing documents in the collection
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    def add_documents(self, documents: List[Document], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        
        ids = []
        metadatas = [] # Initialized as a list
        document_texts = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Create a dictionary for this specific document
            doc_metadata = dict(doc.metadata)
            doc_metadata['doc_index'] = i 
            doc_metadata['content_length'] = len(doc.page_content)
            
            # Append the dictionary to the metadatas list
            metadatas.append(doc_metadata)

            document_texts.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                documents=document_texts,
                metadatas=metadatas,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents.")
        except Exception as e:
            print(f"Error adding documents: {e}")
            raise
vector_store = VectorStore()
vector_store

Vector store initialized with collection: pdf_documents
Existing documents in collection:119


In [12]:
chunks

[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-03-31T15:48:36+05:30', 'source': '..\\data\\Datafile.pdf', 'file_path': '..\\data\\Datafile.pdf', 'total_pages': 90, 'format': 'PDF 1.7', 'title': '', 'author': 'Mounisha Anisetti', 'subject': '', 'keywords': '', 'moddate': '2026-03-31T15:48:36+05:30', 'trapped': '', 'modDate': "D:20260331154836+05'30'", 'creationDate': "D:20260331154836+05'30'", 'page': 0, 'source_file': 'Datafile.pdf', 'file_type': 'pdf'}, page_content='# HR POLICY MANUAL - COMPLETE DOCUMENT FOR RAG MODEL TRAINING \n## Version 1.0 - Anonymized Generic Policies \n \n--- \n \n## TABLE OF CONTENTS \n1. Attendance Policy \n2. Leave Policy   \n3. Salary & Compensation Policy \n4. Travel & Expenses Policy \n5. Employment Terms & Conditions \n6. Performance Management Policy \n7. Disciplinary Conduct Policy \n8. Employee Exit Policy \n9. Grievance Resolution Policy \n10. Training & Development Policy \n \n--- \n 

In [13]:
### Convert the text into embeddings
texts=[doc.page_content for doc in chunks]

##Generate embeddings for the document chunks

embeddings = embedding_manager.generate_embeddings(texts)

##Store in the vector store
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 119 texts...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Generated embeddings with shape: (119, 384)
Successfully added 119 documents.


### Retriever Pipeline from vectorstore

In [14]:
class RAGRetriever:
    #Initialize the retriever
    #Implements retrieval functionality for RAG system using vector store
    """ARGS:
        vector_store: Instance of VectorStore class for managing document chunks and embeddings
        embedding_manager: Instance of EmbeddingManager class for generating query embeddings
    """
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        #Retrieve relevant document chunks based on query
        #Args: query: User query string to retrieve relevant documents for
            #top_k: Number of top relevant documents to return
            #score_threshold: Minimum similarity score for retrieved documents
            #Returns: List of dictionaries containing retrieved document information (content, metadata, similarity score)
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top_k: {top_k}, Score_threshold: {score_threshold}")

        #Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        #Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        #search in vector store for relevant documents
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            #Process results to calculate cosine similarity scores and filter based on score_threshold
            retrieved_docs = []
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                ids=results['ids'][0]
                distance = results['distances'][0]
                for i,(doc_id,document, metadata,distance) in enumerate(zip(ids,documents, metadatas,distance)):
                    #Convert distance to similarity score (assuming distance and chromadb uses cosine distance)
                    similarity_score = 1 - distance
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
            print(f"Retrieved {len(retrieved_docs)} documents after applying score threshold(filtering).")
            return retrieved_docs
        except Exception as e:
            print(f"Error querying vector store: {e}")
            return []
rag_retriever = RAGRetriever(vector_store, embedding_manager)
rag_retriever
        

In [15]:
rag_retriever

In [16]:
rag_retriever.retrieve("What is the Attendance Policy?")

Retrieving documents for query: 'What is the Attendance Policy?'
Top_k: 5, Score_threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents after applying score threshold(filtering).


[{'id': 'doc_b41db6c0_0',
  'content': '# HR POLICY MANUAL - COMPLETE DOCUMENT FOR RAG MODEL TRAINING \n## Version 1.0 - Anonymized Generic Policies \n \n--- \n \n## TABLE OF CONTENTS \n1. Attendance Policy \n2. Leave Policy   \n3. Salary & Compensation Policy \n4. Travel & Expenses Policy \n5. Employment Terms & Conditions \n6. Performance Management Policy \n7. Disciplinary Conduct Policy \n8. Employee Exit Policy \n9. Grievance Resolution Policy \n10. Training & Development Policy \n \n--- \n \n# 1. ATTENDANCE POLICY \n \n## 1.1 Policy Overview \nThis attendance policy establishes guidelines for employee presence, working hours, and attendance \nmanagement across all organizational locations. The policy applies to all full-time and part-time \nemployees. \n \n## 1.2 Working Days and Hours \n \n**Full-Time Employees:** \n- Daily working hours: 8 hours \n- Break time: 1 hour (typically lunch) \n- Total office time: 9 hours per day',
  'metadata': {'doc_index': 0,
   'total_pages': 90,

In [17]:
rag_retriever.retrieve("Absence from Duty")

Retrieving documents for query: 'Absence from Duty'
Top_k: 5, Score_threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents after applying score threshold(filtering).


[{'id': 'doc_e31e48c4_72',
  'content': '- Consistency maintained across similar cases \n \n5. **Legal Considerations:** \n   - Criminal charges: Organization reserves right to take disciplinary action \n   - Legal proceedings: Employment may be affected \n   - Evidence from court/police: Used in disciplinary proceedings \n \n## 7.2 Classification of Offences \n \n### Category 1: Absenteeism \n \n**A. Absenteeism** \n- Being absent from work 10+ consecutive working days \n- Without expressed permission from reporting manager \n- May be ignored if well-justified and approved \n \n**B. Desertion / Absconding** \n- Leaving workplace without intent to return \n- Abandoning work without authority \n- Leaving without permission \n- Assumption of permanent absence \n \n### Category 2: Control and Diligence Offences \n \n**A. Poor Timekeeping** \n- Persistently reporting late for work \n- Persistently leaving early \n- Unauthorized/extended breaks during work \n- Habitual pattern rather than i

### Integration vectordb context pipeline with LLM output

In [36]:
def rag_advanced(query,retriever,llm,top_k=5,min_score=0.2,return_context=False):
    #RAG pipeline with extra features:
    #Returns answer, sources, and optionally retrieved context

    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)

    if not results:
        return{'answer': "No relevant information found in the documents.", 'sources': [], 'confidence': 0, 'context': []}
    
    #Prepare context and sources for LLM
    context = "\n\n".join([doc['content'] for doc in results])
    sources=[{
        'source':doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page':doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview':doc['content'][:120]+'...'
    
    }for doc in results]
    confidence = max(doc['similarity_score'] for doc in results)

    #Generate answer using LLM
    prompt=f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output={
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

#Example usage of the advanced RAG function

result=rag_advanced("What is the Attendance Policy?", rag_retriever, llm, top_k=5, min_score=0.2, return_context=True)
print("Answer:", result['answer'])
print("\nSources:",result['sources'])
print("\nConfidence Score:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is the Attendance Policy?'
Top_k: 5, Score_threshold: 0.2
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents after applying score threshold(filtering).
Answer: The Attendance Policy establishes guidelines for employee presence, working hours, and attendance management across all organizational locations, applying to all full-time and part-time employees.

Sources: [{'source': 'Datafile.pdf', 'page': 0, 'score': 0.21747934818267822, 'preview': '# HR POLICY MANUAL - COMPLETE DOCUMENT FOR RAG MODEL TRAINING \n## Version 1.0 - Anonymized Generic Policies \n \n--- \n \n##...'}, {'source': 'Datafile.pdf', 'page': 0, 'score': 0.21747934818267822, 'preview': '# HR POLICY MANUAL - COMPLETE DOCUMENT FOR RAG MODEL TRAINING \n## Version 1.0 - Anonymized Generic Policies \n \n--- \n \n##...'}]

Confidence Score: 0.21747934818267822
Context Preview: # HR POLICY MANUAL - COMPLETE DOCUMENT FOR RAG MODEL TRAINING 
## Version 1.0 - Anonymized Generic Policies 
 
--- 
 
## TABLE OF CONTENTS 
1. Attendance Policy 
2. Leave Policy   
3. Salary & Comp

In [38]:
import time
from typing import List, Dict, Any
class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> dict:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        
        if not results:
            return {
                "answer": "No relevant context found.",
                "sources": [],
                "context": ""
            }
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [
                {
                    'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                    'page': doc['metadata'].get('page', 'unknown'),
                    'score': doc['similarity_score'],
                    'preview': doc['content'][:120] + '...'
                } for doc in results
            ]
            
            # Define the Prompt answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming response:")
                for i in range(0,len(prompt),80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()  # Newline after streaming
            response = self.llm.invoke([prompt.format(context=context, query=question)])
            answer = response.content

        # Add citations to answer
        citations=[f"[{i+1}] {src['source']} (page{src['page']})" for i,src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations)  if citations else answer

        #Optionally summarize answer if it's too long
        summary=None
        if summarize and answer:
            summary_prompt=f"""Summarize the following answer in 2 sentences:\n{answer}"""
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        #Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            "question": question,
            "answer": answer_with_citations,
            "sources": sources,
            "summary": summary,
            "history": self.history
        }
    

#Example usage
adv_rag=AdvancedRAGPipeline(rag_retriever, llm)
result=adv_rag.query("What is the Attendance Policy?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("Final Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'What is the Attendance Policy?'
Top_k: 3, Score_threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents after applying score threshold(filtering).
Streaming response:
Use the following context to answer the question concisely.
Context:
# HR POLICY MANUAL - COMPLETE DOCUMENT FOR RAG MODEL TRAINING 
## Version 1.0 - Anonymized Generic Policies 
 
--- 
 
## TABLE OF CONTENTS 
1. Attendance Policy 
2. Leave Policy   
3. Salary & Compensation Policy 
4. Travel & Expenses Policy 
5. Employment Terms & Conditions 
6. Performance Management Policy 
7. Disciplinary Conduct Policy 
8. Employee Exit Policy 
9. Grievance Resolution Policy 
10. Training & Development Policy 
 
--- 
 
# 1. ATTENDANCE POLICY 
 
## 1.1 Policy Overview 
This attendance policy establishes guidelines for employee presence, working hours, and attendance 
management across all organizational locations. The policy applies to all full-time and part-time 
employees. 
 
## 1.2 Working Days and Hours 
 
**Full-Time Employees:** 
- Daily working hours: 8 hours 
- Brea